In [125]:
import sys
from pathlib import Path
sys.path[:0] = [str(Path.cwd().parent)]

import numpy as np

from constants import *
from self_consistency_k import build_bdg_hamiltonian, bdg_sc_full_k

In [154]:
t=1
mu=1.3
T = 0.05

U=0
V=1.5
V_prime=1.5

h = np.array([0, 0, 0])
Nx, Ny = 30, 30

out = bdg_sc_full_k(
    t, mu, Nx, Ny, V=V,
    temperature=T, maxiter=5000,
    verbose=False, atol=1e-6,rtol=1e-3,
    F_init=np.array([0.1+0.1, 0.1-0.1, -0.1, -0.1], dtype=np.complex128),
)

In [155]:
print(f"onsite: {out.F_onsite}")
print(f"swave: {out.F_swave}")
print(f"dwave: {out.F_dwave}")
print(f"px: {out.F_px}")
print(f"py: {out.F_py}")
print(f"Free energy: {out.free_energy}")

onsite: (1.0422740415964166e-18+3.231854210248783e-18j)
swave: (-2.1978387398880958e-18+8.673617379884035e-19j)
dwave: (1.9545073499656167e-17-2.168404344971009e-17j)
px: (0.018723493134996185-9.7671792664367e-17j)
py: (-8.16641405106816e-17-0.01862164210037151j)
Free energy: -1771.542257136398


In [128]:
def stable_config(out, atol=1e-6, rtol=0.1):
    stable = {}
    F_max = out.corr[np.argmax(np.abs(out.corr))]
    if np.abs(F_max) < atol:
        return stable
    for i, c in enumerate(out[1:-3]):
        if np.abs(c) > atol and np.abs(c) > rtol * np.abs(F_max):
            stable[corr_strings[i]] = c
    return stable


def same_stable_dict(d1, d2, atol=1e-6, rtol=1e-3):
    if d1.keys() != d2.keys():
        return False

    for k in d1:
        if not np.isclose(d1[k], d2[k], atol=atol, rtol=rtol):
            return False

    return True


In [129]:
t=1
mu=0.95
T = 0.0001

U=0
V=1.5
V_prime=1.5

h = np.array([0, 0, 0])
Nx, Ny = 30, 30

In [130]:
initial_seeds = [
    # np.zeros(4, dtype=np.complex128),
    np.array([0.1, 0.1, -0.1, -0.1], dtype=np.complex128),
    np.array([0.1+0.1, 0.1-0.1, -0.1, -0.1], dtype=np.complex128),
    np.array([0.1, 0.1, -0.1+0.1, -0.1-0.1], dtype=np.complex128),

]

seed_strings = [
    # "normal", 
    "d", "d+px", "d+py"
]

atol = 1e-8
rtol = 1e-6
maxiter=5000
free_tol =0.01

best_free = np.inf
configs = []

print("====================================")
print(f"mu={mu:.2f}, T={T:.4f}")
print("====================================")
for seed, seed_str in zip(initial_seeds, seed_strings):
    print(f"  Seed: {seed_str}")

    out = bdg_sc_full_k(
        t, mu, temperature=T,
        V=V, Nx=Nx, Ny=Ny,
        atol=atol, rtol=rtol,
        maxiter=maxiter,
        F_init=seed
    )
    print(f"onsite: {out.F_onsite}")
    print(f"swave: {out.F_swave}")
    print(f"dwave: {out.F_dwave}")
    print(f"px: {out.F_px}")
    print(f"py: {out.F_py}")
    print(f"Free energy: {out.free_energy}")
    stable = stable_config(out, atol=atol,rtol=rtol)

    print("------------------------------------------")
    if (out.free_energy < best_free and
            np.abs(out.free_energy - best_free) > free_tol):

        best_free = out.free_energy
        configs = [{
            "stable": stable,
            "free": out.free_energy,
        }]

    elif np.abs(out.free_energy - best_free) <= free_tol:
        if len(stable) != 0 and not any(
            same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
            for c in configs
        ):
            configs.append({
                "stable": stable,
                "free": out.free_energy,
            })

    print(configs)
    print("------------------------------------------")

records = {
    "mu": mu,
    "T": T,
    "best_free": best_free,
    "configs": configs
}


mu=0.95, T=0.0001
  Seed: d
onsite: (-8.637406721388914e-18+0j)
swave: (-3.469446951953614e-18+0j)
dwave: (0.050057531331276134+0j)
px: 5.176165401111826e-18j
py: -1.055445403926221e-17j
Free energy: -1652.898271343229
------------------------------------------
[{'stable': {'F_dwave': np.complex128(0.050057531331276134+0j)}, 'free': np.float64(-1652.898271343229)}]
------------------------------------------
  Seed: d+px
onsite: (-2.2135088494123004e-18+1.3864510078928547e-18j)
swave: (6.938893903907228e-18-5.772000139946023e-19j)
dwave: (0.050057531331276064+7.280777577242745e-19j)
px: (6.938893903907228e-18+3.1130313696167656e-17j)
py: (-6.938893903907228e-18-1.482360597254229e-17j)
Free energy: -1652.8982713432288
------------------------------------------
[{'stable': {'F_dwave': np.complex128(0.050057531331276134+0j)}, 'free': np.float64(-1652.898271343229)}]
------------------------------------------
  Seed: d+py
onsite: (-1.272772882367993e-17+7.345491000385179e-19j)
swave: -1.133

In [131]:
print(records)

{'mu': 0.95, 'T': 0.0001, 'best_free': np.float64(-1652.898271343229), 'configs': [{'stable': {'F_dwave': np.complex128(0.050057531331276134+0j)}, 'free': np.float64(-1652.898271343229)}]}


In [132]:
import pandas as pd

In [133]:
def flatten_df(df):
    df_flat = df.copy()

    # Make sure configs is always a list before exploding
    df_flat["configs"] = df_flat["configs"].apply(
        lambda x: x if isinstance(x, list)
        else [x] if isinstance(x, dict)
        else []
    )

    df_flat = df_flat.explode("configs").reset_index(drop=True)

    # Split out the free energy and stable dict
    df_flat["free"] = df_flat["configs"].apply(
        lambda x: x.get("free", 0) if isinstance(x, dict) else 0
    )
    df_flat["stable"] = df_flat["configs"].apply(
        lambda x: x.get("stable", {}) if isinstance(x, dict) else {}
    )

    # Expand the stable dictionaries into columns
    stable_df = pd.DataFrame(df_flat["stable"].tolist())

    # Ensure all desired columns exist
    stable_df = stable_df.reindex(columns=corr_strings, fill_value=0)

    # Combine everything
    df_flat = pd.concat(
        [
            df_flat.drop(columns=["configs", "stable"]),
            stable_df
        ],
        axis=1
    )

    df_flat.fillna(0, inplace=True)
    return df_flat

In [134]:
df = pd.DataFrame(records)
df.head()

,mu,T,best_free,configs
0,0.95,0.0001,-1652.898271,{'stable': {'F_dwave': (0.050057531331276134+0...


In [135]:
df_flat = flatten_df(df)

In [136]:
df_flat.head()

,mu,T,best_free,free,F_onsite,F_swave,F_dwave,F_px,F_py,Fuu_px,Fuu_py,Fdd_px,Fdd_py
0,0.95,0.0001,-1652.898271,-1652.898271,0,0,0.050058+0.000000j,0,0,0,0,0,0


In [137]:
df.iloc[0]["configs"]

{'stable': {'F_dwave': np.complex128(0.050057531331276134+0j)},
 'free': np.float64(-1652.898271343229)}